# Kabadiwala Connect: AI/ML Layer (PyTorch)

Two-stage vision pipeline for waste imagery.

| Stage | Class | Backbone | Output |
|---|---|---|---|
| Pipeline 1 | `WasteSegregationCNN` | MobileNetV3-Small / EfficientNet-B0 | `0 = MixedPlastics`, `1 = E_Waste` |
| Pipeline 2 | `EwasteValuationMultiHeadCNN` | EfficientNet-B4 (shared) | `category`, `subCategory`, condition, materials, `estimatedValue` |

**Data-dictionary alignment**

| Dictionary field | Backend JSON (snake_case) | Client / Dart state (camelCase) |
|---|---|---|
| category | `material_category` | `category` |
| subCategory | `sub_category` | `subCategory` |
| approxWeightKg | `approx_weight_kg` | `approxWeightKg` |
| estimatedValue | `estimated_value` | `estimatedValue` |

**Design notes**
- `estimatedValue` is regressed in `log1p(INR)` space and converted back to ₹ at inference.
- The value head also sees the (detached) predicted category / sub-category / condition / material probabilities and the user-entered weight.
- Missing labels are encoded with `IGNORE_INDEX` / masks so partially labelled records still train.
- `DEFAULT_WEIGHT_PRIOR_KG` and `mixed_plastics_rate_inr_per_kg` are **placeholders**. Tune them from real data.

Run the cells top to bottom. The last section is a smoke test that works offline (`pretrained=False`).

## 0. Setup

In [57]:
# Uncomment to install dependencies (torch/torchvision usually come pre-installed on Colab)
# %pip install -q torch torchvision albumentations opencv-python-headless numpy pillow


In [58]:
from __future__ import annotations

import json
import random
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple, Union

import albumentations as A
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import Dataset
from torchvision import models


## 1. Schema / enum constants (must match the Kabadiwala Connect Data Dictionary)

In [59]:
IMAGENET_MEAN: Tuple[float, float, float] = (0.485, 0.456, 0.406)
IMAGENET_STD: Tuple[float, float, float] = (0.229, 0.224, 0.225)
IGNORE_INDEX: int = -100

# Pipeline 1 output classes
SEGREGATION_LABELS: Dict[int, str] = {0: "MixedPlastics", 1: "E_Waste"}

# Data-dictionary `category` enum (full) and the subset Pipeline 2 predicts
ALL_CATEGORIES: List[str] = ["PCB", "CRT", "Cables", "Battery", "Motor", "MixedPlastics"]
EWASTE_CATEGORIES: List[str] = ["PCB", "CRT", "Cables", "Battery", "Motor"]

# `subCategory` head labels
SUB_CATEGORIES: List[str] = [
    "Motherboard",
    "GPU_Board",
    "Power_Supply",
    "Li_Ion_Cell",
    "Copper_Cable",
    "General_Ewaste",
]
GENERIC_SUB_CATEGORY: str = "General_Ewaste"

CONDITIONS: List[str] = ["Intact", "Minor_Damage", "Scrap_Only"]

MATERIALS: List[str] = [
    "Copper",
    "Aluminum",
    "Gold_Plated_Connectors",
    "Lithium",
    "Steel_Iron",
    "Precious_Metals_PCB",
]

# Which sub-categories are plausible for each category. Used at inference to
# suppress contradictory predictions (e.g. category=Battery + Motherboard).
CATEGORY_TO_SUB_CATEGORIES: Dict[str, List[str]] = {
    "PCB": ["Motherboard", "GPU_Board", "Power_Supply", "General_Ewaste"],
    "CRT": ["General_Ewaste"],
    "Cables": ["Copper_Cable", "General_Ewaste"],
    "Battery": ["Li_Ion_Cell", "General_Ewaste"],
    "Motor": ["General_Ewaste"],
}

# PLACEHOLDER typical unit weights (kg), used only when the user gives no weight.
DEFAULT_WEIGHT_PRIOR_KG: Dict[str, float] = {
    "PCB": 0.5,
    "CRT": 10.0,
    "Cables": 1.0,
    "Battery": 0.1,
    "Motor": 2.0,
    "MixedPlastics": 1.0,
}

_CATEGORY_TO_IDX = {c: i for i, c in enumerate(EWASTE_CATEGORIES)}
_SUB_TO_IDX = {c: i for i, c in enumerate(SUB_CATEGORIES)}
_COND_TO_IDX = {c: i for i, c in enumerate(CONDITIONS)}
_MAT_TO_IDX = {c: i for i, c in enumerate(MATERIALS)}


## 2. Value-space helpers (INR <-> log1p(INR))

In [60]:
def inr_to_log(value_inr: torch.Tensor) -> torch.Tensor:
    """Map a price in INR (>= 0) to ``log1p`` space used for regression."""
    return torch.log1p(value_inr.clamp(min=0.0))


def log_to_inr(value_log: torch.Tensor) -> torch.Tensor:
    """Inverse of :func:`inr_to_log`; output is clamped to ``>= 0`` INR."""
    return torch.expm1(value_log.clamp(max=20.0)).clamp(min=0.0)


## 3. Augmentation

In [61]:
class BackgroundBlur(A.ImageOnlyTransform):
    """
    Blur everything outside a soft, centred elliptical "focus" region.

    Simulates shallow depth-of-field / cluttered kabadiwala-shop backgrounds
    without needing segmentation masks (the object is assumed to be roughly
    centred). The focus region edge is feathered to avoid hard seams.
    """

    def __init__(
        self,
        blur_limit: Tuple[int, int] = (15, 41),
        focus_ratio: Tuple[float, float] = (0.55, 0.85),
        p: float = 0.3,
    ) -> None:
        super().__init__(p=p)
        self.blur_limit = blur_limit
        self.focus_ratio = focus_ratio

    def apply(self, img: np.ndarray, **params: Any) -> np.ndarray:
        h, w = img.shape[:2]
        kernel = random.randint(*self.blur_limit) | 1  # force odd kernel size
        blurred = cv2.GaussianBlur(img, (kernel, kernel), 0)

        ratio = random.uniform(*self.focus_ratio)
        mask = np.zeros((h, w), dtype=np.float32)
        cv2.ellipse(
            mask, (w // 2, h // 2), (int(w * ratio / 2), int(h * ratio / 2)),
            0, 0, 360, 1.0, -1,
        )
        mask = cv2.GaussianBlur(mask, (0, 0), sigmaX=max(h, w) * 0.05)[..., None]
        out = img.astype(np.float32) * mask + blurred.astype(np.float32) * (1.0 - mask)
        return np.clip(out, 0, 255).astype(np.uint8)

    def get_transform_init_args_names(self) -> Tuple[str, ...]:
        return ("blur_limit", "focus_ratio")


def build_train_transforms(image_size: int = 380) -> A.Compose:
    """Training augmentation: random crop, rotation, brightness/contrast, background blur."""
    return A.Compose(
        [
            A.SmallestMaxSize(max_size=int(image_size * 1.15)),
            A.RandomCrop(height=image_size, width=image_size),
            A.Rotate(limit=25, border_mode=cv2.BORDER_REFLECT_101, p=0.6),
            A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),
            BackgroundBlur(p=0.3),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]
    )


def build_eval_transforms(image_size: int = 380) -> A.Compose:
    """Deterministic validation / inference preprocessing."""
    return A.Compose(
        [
            A.Resize(image_size, image_size),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]
    )


## 4. Dataset

In [62]:
class EwasteDataset(Dataset):
    """
    Multi-task dataset for Pipeline 2, aligned with the backend JSON fields.

    Each record is a ``dict`` (snake_case, as served by the REST API)::

        {
          "image_path": "imgs/0001.jpg",
          "material_category": "PCB",          # or "category"; one of EWASTE_CATEGORIES
          "sub_category": "Motherboard",        # optional
          "condition": "Intact",                # optional
          "materials": ["Copper", "Gold_Plated_Connectors"],   # optional (names or multi-hot)
          "approx_weight_kg": 0.85,             # optional
          "estimated_value": 240.0              # optional, INR
        }

    Missing optional labels are converted to ignore-values / masks so that the
    loss (:class:`EwasteMultiTaskLoss`) skips them.

    ``__getitem__`` returns a dict of tensors:
    ``image, category, sub_category, condition, materials, material_mask,
    approx_weight_kg, has_weight, estimated_value, value_mask``.
    """

    def __init__(
        self,
        records: Sequence[Dict[str, Any]],
        image_root: Optional[Union[str, Path]] = None,
        transform: Optional[Callable[..., Dict[str, torch.Tensor]]] = None,
        image_size: int = 380,
        training: bool = True,
    ) -> None:
        self.records = list(records)
        self.image_root = Path(image_root) if image_root is not None else None
        self.transform = transform or (
            build_train_transforms(image_size) if training else build_eval_transforms(image_size)
        )

    @classmethod
    def from_json(cls, json_path: Union[str, Path], **kwargs: Any) -> "EwasteDataset":
        """Build from a JSON file containing a list of records."""
        with open(json_path, "r", encoding="utf-8") as fh:
            return cls(json.load(fh), **kwargs)

    def __len__(self) -> int:
        return len(self.records)

    # -- helpers -------------------------------------------------------------
    @staticmethod
    def _first(record: Dict[str, Any], *keys: str) -> Any:
        for key in keys:
            if record.get(key) is not None:
                return record[key]
        return None

    def _load_image(self, path: Union[str, Path]) -> np.ndarray:
        path = Path(path)
        if self.image_root is not None and not path.is_absolute():
            path = self.image_root / path
        with Image.open(path) as img:
            return np.asarray(img.convert("RGB"))

    @staticmethod
    def _encode_materials(raw: Any) -> Tuple[torch.Tensor, float]:
        """Return (multi-hot vector, mask) where mask=0 means 'label unknown'."""
        vec = torch.zeros(len(MATERIALS), dtype=torch.float32)
        if raw is None:
            return vec, 0.0
        if len(raw) > 0 and isinstance(raw[0], str):
            for name in raw:
                if name not in _MAT_TO_IDX:
                    raise ValueError(f"Unknown material '{name}'. Expected one of {MATERIALS}")
                vec[_MAT_TO_IDX[name]] = 1.0
        else:
            if len(raw) != len(MATERIALS):
                raise ValueError(f"Multi-hot materials must have length {len(MATERIALS)}")
            vec = torch.tensor(raw, dtype=torch.float32)
        return vec, 1.0

    # -- main ----------------------------------------------------------------
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        rec = self.records[idx]

        image = self._load_image(rec["image_path"])
        image_t = self.transform(image=image)["image"]

        category = self._first(rec, "material_category", "category")
        if category not in _CATEGORY_TO_IDX:
            raise ValueError(f"Record {idx}: category '{category}' not in {EWASTE_CATEGORIES}")

        sub = self._first(rec, "sub_category", "subCategory")
        cond = self._first(rec, "condition", "physical_condition")
        if sub is not None and sub not in _SUB_TO_IDX:
            raise ValueError(f"Record {idx}: unknown sub_category '{sub}'")
        if cond is not None and cond not in _COND_TO_IDX:
            raise ValueError(f"Record {idx}: unknown condition '{cond}'")

        materials, material_mask = self._encode_materials(rec.get("materials"))

        weight = self._first(rec, "approx_weight_kg", "approxWeightKg")
        value = self._first(rec, "estimated_value", "estimatedValue")

        return {
            "image": image_t,
            "category": torch.tensor(_CATEGORY_TO_IDX[category], dtype=torch.long),
            "sub_category": torch.tensor(
                _SUB_TO_IDX[sub] if sub is not None else IGNORE_INDEX, dtype=torch.long
            ),
            "condition": torch.tensor(
                _COND_TO_IDX[cond] if cond is not None else IGNORE_INDEX, dtype=torch.long
            ),
            "materials": materials,
            "material_mask": torch.tensor(material_mask, dtype=torch.float32),
            "approx_weight_kg": torch.tensor(float(weight) if weight is not None else 0.0),
            "has_weight": torch.tensor(1.0 if weight is not None else 0.0),
            "estimated_value": torch.tensor(float(value) if value is not None else 0.0),
            "value_mask": torch.tensor(1.0 if value is not None else 0.0),
        }


## 5. Pipeline 1: binary segregation

In [63]:
class WasteSegregationCNN(nn.Module):
    """
    Fast binary screening model: ``0 = MixedPlastics``, ``1 = E_Waste``.

    Args:
        backbone: ``"mobilenet_v3_small"`` (default, fastest) or ``"efficientnet_b0"``.
        pretrained: Load ImageNet weights (needs network / cached weights).
        dropout: Dropout probability before the final classifier layer.
        freeze_backbone: Freeze the convolutional feature extractor (linear-probe mode).
    """

    NUM_CLASSES: int = 2

    def __init__(
        self,
        backbone: str = "mobilenet_v3_small",
        pretrained: bool = True,
        dropout: float = 0.2,
        freeze_backbone: bool = False,
    ) -> None:
        super().__init__()
        self.backbone_name = backbone

        if backbone == "mobilenet_v3_small":
            weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
            net = models.mobilenet_v3_small(weights=weights)
            in_features = net.classifier[3].in_features
            net.classifier[2] = nn.Dropout(p=dropout)
            net.classifier[3] = nn.Linear(in_features, self.NUM_CLASSES)
        elif backbone == "efficientnet_b0":
            weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
            net = models.efficientnet_b0(weights=weights)
            in_features = net.classifier[1].in_features
            net.classifier = nn.Sequential(
                nn.Dropout(p=dropout), nn.Linear(in_features, self.NUM_CLASSES)
            )
        else:
            raise ValueError("backbone must be 'mobilenet_v3_small' or 'efficientnet_b0'")

        if freeze_backbone:
            for p in net.features.parameters():
                p.requires_grad = False
        self.net = net

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """Return raw logits of shape ``(B, 2)``."""
        return self.net(images)

    @torch.no_grad()
    def predict_proba(self, images: torch.Tensor) -> torch.Tensor:
        """Return class probabilities ``(B, 2)``; column 1 is P(E_Waste)."""
        return F.softmax(self.forward(images), dim=-1)


## 6. Pipeline 2: multi-head characterisation & valuation

In [64]:
def _make_head(in_dim: int, hidden: int, out_dim: int, dropout: float) -> nn.Sequential:
    """Small MLP head: Dropout -> Linear -> ReLU -> Dropout -> Linear."""
    return nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_dim, hidden),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout),
        nn.Linear(hidden, out_dim),
    )


class EwasteValuationMultiHeadCNN(nn.Module):
    """
    Shared EfficientNet-B4 trunk with five task heads.

    Heads (attribute names are part of the checkpoint contract):
        * ``category_head``        -> logits over ``EWASTE_CATEGORIES`` (5)
        * ``sub_category_head``    -> logits over ``SUB_CATEGORIES`` (6)
        * ``condition_head``       -> logits over ``CONDITIONS`` (3)
        * ``material_head``        -> multi-label logits over ``MATERIALS`` (6)
        * ``estimated_value_head`` -> scalar ``log1p(INR)`` regression

    The value head receives ``[pooled features | softmax(category) |
    softmax(sub_category) | softmax(condition) | sigmoid(material) |
    log1p(weight) | has_weight]``. Auxiliary probabilities are detached so the
    (noisier) price loss does not distort the classification heads.

    Args:
        pretrained: Load ImageNet weights for EfficientNet-B4.
        head_hidden: Hidden width of each head.
        dropout: Dropout inside heads.
        weight_dropout_p: During training, probability of hiding the weight input
            so the model also works when the user does not enter a weight.
        freeze_backbone: Freeze the EfficientNet-B4 feature extractor.
    """

    def __init__(
        self,
        pretrained: bool = True,
        head_hidden: int = 512,
        dropout: float = 0.3,
        weight_dropout_p: float = 0.3,
        freeze_backbone: bool = False,
    ) -> None:
        super().__init__()
        weights = models.EfficientNet_B4_Weights.IMAGENET1K_V1 if pretrained else None
        net = models.efficientnet_b4(weights=weights)

        self.features = net.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.feature_dim: int = net.classifier[1].in_features  # 1792 for B4
        self.weight_dropout_p = weight_dropout_p

        if freeze_backbone:
            for p in self.features.parameters():
                p.requires_grad = False

        n_cat, n_sub = len(EWASTE_CATEGORIES), len(SUB_CATEGORIES)
        n_cond, n_mat = len(CONDITIONS), len(MATERIALS)

        self.category_head = _make_head(self.feature_dim, head_hidden, n_cat, dropout)
        self.sub_category_head = _make_head(self.feature_dim, head_hidden, n_sub, dropout)
        self.condition_head = _make_head(self.feature_dim, head_hidden, n_cond, dropout)
        self.material_head = _make_head(self.feature_dim, head_hidden, n_mat, dropout)

        value_in = self.feature_dim + n_cat + n_sub + n_cond + n_mat + 2  # +log-weight, +has_weight
        self.estimated_value_head = nn.Sequential(
            nn.Linear(value_in, head_hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(head_hidden, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1),
        )

    def forward(
        self,
        images: torch.Tensor,
        approx_weight_kg: Optional[torch.Tensor] = None,
        has_weight: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        """
        Args:
            images: ``(B, 3, H, W)`` normalised images (any H, W >= 32; 380 recommended).
            approx_weight_kg: Optional ``(B,)`` weights in kg. NaN entries are treated as unknown.
            has_weight: Optional ``(B,)`` 0/1 flags; defaults to "known" wherever a weight is given.

        Returns:
            Dict with keys ``category``, ``sub_category``, ``condition``, ``material``
            (all logits) and ``estimated_value_log`` (``(B,)``, log1p-INR space).
        """
        feats = self.pool(self.features(images)).flatten(1)  # (B, feature_dim)
        b = feats.size(0)

        category_logits = self.category_head(feats)
        sub_logits = self.sub_category_head(feats)
        cond_logits = self.condition_head(feats)
        material_logits = self.material_head(feats)

        # ---- weight conditioning -------------------------------------------
        if approx_weight_kg is None:
            weight = feats.new_zeros(b, 1)
            known = feats.new_zeros(b, 1)
        else:
            weight = approx_weight_kg.to(feats.device, feats.dtype).view(b, 1)
            known = (~torch.isnan(weight)).to(feats.dtype)
            if has_weight is not None:
                known = known * has_weight.to(feats.device, feats.dtype).view(b, 1)
            if self.training and self.weight_dropout_p > 0:
                keep = (torch.rand(b, 1, device=feats.device) >= self.weight_dropout_p).to(feats.dtype)
                known = known * keep
            weight = torch.nan_to_num(weight, nan=0.0).clamp(min=0.0) * known

        value_input = torch.cat(
            [
                feats,
                F.softmax(category_logits, dim=-1).detach(),
                F.softmax(sub_logits, dim=-1).detach(),
                F.softmax(cond_logits, dim=-1).detach(),
                torch.sigmoid(material_logits).detach(),
                torch.log1p(weight),
                known,
            ],
            dim=-1,
        )
        value_log = self.estimated_value_head(value_input).squeeze(-1)

        return {
            "category": category_logits,
            "sub_category": sub_logits,
            "condition": cond_logits,
            "material": material_logits,
            "estimated_value_log": value_log,
        }


## 7. Multi-task loss

In [65]:
class EwasteMultiTaskLoss(nn.Module):
    r"""
    .. math::
        \mathcal{L}_{total} = \alpha\mathcal{L}_{cat} + \beta\mathcal{L}_{subCat}
        + \gamma\mathcal{L}_{cond} + \delta\mathcal{L}_{mat} + \epsilon\mathcal{L}_{value}

    * ``L_cat``, ``L_subCat``, ``L_cond``: cross-entropy (``ignore_index`` skips unlabeled rows).
    * ``L_mat``: BCE-with-logits, averaged over labelled samples only (``material_mask``).
    * ``L_value``: Smooth-L1 between predicted ``log1p`` value and ``log1p(estimated_value)``,
      averaged over samples with a value label (``value_mask``).

    Args:
        alpha, beta, gamma, delta, epsilon: Task weights.
        label_smoothing: Label smoothing for the CE terms.
        smooth_l1_beta: Transition point of Smooth-L1.
        material_pos_weight: Optional ``(6,)`` tensor to up-weight rare materials.
    """

    def __init__(
        self,
        alpha: float = 1.0,
        beta: float = 0.7,
        gamma: float = 0.7,
        delta: float = 1.0,
        epsilon: float = 0.5,
        label_smoothing: float = 0.05,
        smooth_l1_beta: float = 1.0,
        material_pos_weight: Optional[torch.Tensor] = None,
    ) -> None:
        super().__init__()
        self.alpha, self.beta, self.gamma = alpha, beta, gamma
        self.delta, self.epsilon = delta, epsilon
        self.label_smoothing = label_smoothing
        self.smooth_l1_beta = smooth_l1_beta
        self.register_buffer("material_pos_weight", material_pos_weight)

    def _masked_ce(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        if not (target != IGNORE_INDEX).any():
            return logits.sum() * 0.0  # keeps the autograd graph valid
        return F.cross_entropy(
            logits, target, ignore_index=IGNORE_INDEX, label_smoothing=self.label_smoothing
        )

    def forward(
        self, outputs: Dict[str, torch.Tensor], targets: Dict[str, torch.Tensor]
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        Args:
            outputs: Dict from :class:`EwasteValuationMultiHeadCNN`.
            targets: Batch dict from :class:`EwasteDataset` (on the same device).

        Returns:
            ``(total_loss, components)`` where ``components`` holds detached scalars
            (``category, sub_category, condition, material, value, total``) for logging.
        """
        l_cat = self._masked_ce(outputs["category"], targets["category"])
        l_sub = self._masked_ce(outputs["sub_category"], targets["sub_category"])
        l_cond = self._masked_ce(outputs["condition"], targets["condition"])

        mat = F.binary_cross_entropy_with_logits(
            outputs["material"],
            targets["materials"],
            pos_weight=self.material_pos_weight,
            reduction="none",
        )
        mat_mask = targets["material_mask"].view(-1, 1)
        l_mat = (mat * mat_mask).sum() / (mat_mask.sum() * mat.size(1)).clamp(min=1.0)

        val = F.smooth_l1_loss(
            outputs["estimated_value_log"],
            inr_to_log(targets["estimated_value"]),
            beta=self.smooth_l1_beta,
            reduction="none",
        )
        val_mask = targets["value_mask"]
        l_val = (val * val_mask).sum() / val_mask.sum().clamp(min=1.0)

        total = (
            self.alpha * l_cat
            + self.beta * l_sub
            + self.gamma * l_cond
            + self.delta * l_mat
            + self.epsilon * l_val
        )
        parts = {
            "category": l_cat.detach(),
            "sub_category": l_sub.detach(),
            "condition": l_cond.detach(),
            "material": l_mat.detach(),
            "value": l_val.detach(),
            "total": total.detach(),
        }
        return total, parts


## 8. Inference engine

In [66]:
@dataclass
class EngineConfig:
    """
    Runtime configuration for :class:`KabadiwalaAIInferenceEngine`.

    Attributes:
        segregation_image_size: Input size for Pipeline 1.
        valuation_image_size: Input size for Pipeline 2 (EfficientNet-B4 native = 380).
        e_waste_threshold: Route to Pipeline 2 when ``P(E_Waste) >= threshold``.
        material_threshold: Sigmoid cut-off for reporting a material as detected.
        sub_category_min_confidence: Below this, ``sub_category`` is returned as ``None``.
        null_generic_sub_category: Serialise ``General_Ewaste`` as ``None`` (it is not a finer class).
        mixed_plastics_rate_inr_per_kg: Optional rule-based rate; ``None`` -> ``estimated_value = None``.
        default_weight_prior_kg: Placeholder weights used when the user gives none.
    """

    segregation_image_size: int = 224
    valuation_image_size: int = 380
    e_waste_threshold: float = 0.5
    material_threshold: float = 0.5
    sub_category_min_confidence: float = 0.35
    null_generic_sub_category: bool = True
    mixed_plastics_rate_inr_per_kg: Optional[float] = None
    default_weight_prior_kg: Dict[str, float] = field(
        default_factory=lambda: dict(DEFAULT_WEIGHT_PRIOR_KG)
    )


def _snake_to_camel(name: str) -> str:
    head, *rest = name.split("_")
    return head + "".join(part.capitalize() for part in rest)


# Backend `material_category` maps to the client's `category` field.
_CAMEL_OVERRIDES: Dict[str, str] = {"material_category": "category"}


def _to_camel_keys(obj: Any) -> Any:
    """Recursively convert dict keys from snake_case to Dart-style camelCase."""
    if isinstance(obj, dict):
        return {
            _CAMEL_OVERRIDES.get(k, _snake_to_camel(k)): _to_camel_keys(v) for k, v in obj.items()
        }
    if isinstance(obj, list):
        return [_to_camel_keys(v) for v in obj]
    return obj


class KabadiwalaAIInferenceEngine:
    """
    End-to-end inference: segregation -> (optional) characterisation & valuation.

    Args:
        segregation_model: Trained :class:`WasteSegregationCNN`.
        valuation_model: Trained :class:`EwasteValuationMultiHeadCNN`.
        config: :class:`EngineConfig` (defaults used when ``None``).
        device: Target device; defaults to CUDA when available, else CPU.
    """

    def __init__(
        self,
        segregation_model: WasteSegregationCNN,
        valuation_model: EwasteValuationMultiHeadCNN,
        config: Optional[EngineConfig] = None,
        device: Optional[torch.device] = None,
    ) -> None:
        self.config = config or EngineConfig()
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.segregation_model = segregation_model.to(self.device).eval()
        self.valuation_model = valuation_model.to(self.device).eval()
        self._transforms: Dict[int, A.Compose] = {}

    @classmethod
    def from_checkpoints(
        cls,
        segregation_ckpt: Optional[Union[str, Path]] = None,
        valuation_ckpt: Optional[Union[str, Path]] = None,
        config: Optional[EngineConfig] = None,
        device: Optional[torch.device] = None,
        segregation_backbone: str = "mobilenet_v3_small",
    ) -> "KabadiwalaAIInferenceEngine":
        """Build the engine and load ``state_dict`` checkpoints (skipped when ``None``)."""
        device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        seg = WasteSegregationCNN(backbone=segregation_backbone, pretrained=False)
        val = EwasteValuationMultiHeadCNN(pretrained=False)
        if segregation_ckpt is not None:
            seg.load_state_dict(torch.load(segregation_ckpt, map_location=device))
        if valuation_ckpt is not None:
            val.load_state_dict(torch.load(valuation_ckpt, map_location=device))
        return cls(seg, val, config=config, device=device)

    # -- input handling --------------------------------------------------------
    @staticmethod
    def _load_raw(image: Union[str, Path, Image.Image, np.ndarray, torch.Tensor]) -> Union[np.ndarray, torch.Tensor]:
        """Normalise supported input types to an RGB uint8 array or a float tensor."""
        if isinstance(image, torch.Tensor):
            return image
        if isinstance(image, (str, Path)):
            image = Image.open(image)
        if isinstance(image, Image.Image):
            return np.asarray(image.convert("RGB"))
        if isinstance(image, np.ndarray):
            if image.dtype != np.uint8:
                raise ValueError("numpy images must be uint8 RGB (H, W, 3)")
            if image.ndim == 2:
                image = np.stack([image] * 3, axis=-1)
            return image[..., :3]
        raise TypeError(f"Unsupported image type: {type(image)}")

    def _to_tensor(self, raw: Union[np.ndarray, torch.Tensor], size: int) -> torch.Tensor:
        """Return a ``(1, 3, size, size)`` normalised tensor on ``self.device``."""
        if isinstance(raw, torch.Tensor):
            t = raw.detach().float()  # tensors are assumed to be already normalised
            if t.ndim == 3:
                t = t.unsqueeze(0)
            if t.ndim != 4 or t.shape[0] != 1 or t.shape[1] != 3:
                raise ValueError(f"Tensor input must be (3,H,W) or (1,3,H,W); got {tuple(t.shape)}")
            if tuple(t.shape[-2:]) != (size, size):
                t = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)
            return t.to(self.device)
        if size not in self._transforms:
            self._transforms[size] = build_eval_transforms(size)
        return self._transforms[size](image=raw)["image"].unsqueeze(0).to(self.device)

    @staticmethod
    def _validate_weight(weight: Optional[float]) -> Optional[float]:
        if weight is None:
            return None
        weight = float(weight)
        if not np.isfinite(weight) or weight <= 0:
            raise ValueError("approx_weight_kg must be a positive, finite number")
        return weight

    # -- main entry point --------------------------------------------------------
    @torch.inference_mode()
    def predict(
        self,
        image: Union[str, Path, Image.Image, np.ndarray, torch.Tensor],
        approx_weight_kg: Optional[float] = None,
    ) -> Dict[str, Any]:
        """
        Run the full pipeline on one image.

        Args:
            image: File path, PIL image, RGB ``uint8`` array, or an already-normalised
                tensor ``(3,H,W)`` / ``(1,3,H,W)``.
            approx_weight_kg: Optional user-entered weight (``approxWeightKg``).

        Returns:
            ``{"rest_api": {...snake_case...}, "dart_ui_state": {...camelCase...}}``
        """
        cfg = self.config
        user_weight = self._validate_weight(approx_weight_kg)
        raw = self._load_raw(image)

        # ---- Pipeline 1: segregation ----------------------------------------
        x1 = self._to_tensor(raw, cfg.segregation_image_size)
        seg_probs = F.softmax(self.segregation_model(x1), dim=-1)[0]
        p_ewaste = float(seg_probs[1])

        if p_ewaste < cfg.e_waste_threshold:
            return self._mixed_plastics_response(user_weight, 1.0 - p_ewaste, p_ewaste)

        # ---- Pipeline 2: characterisation & valuation -----------------------
        x2 = self._to_tensor(raw, cfg.valuation_image_size)
        weight_t = None if user_weight is None else torch.tensor([user_weight], device=self.device)
        out = self.valuation_model(x2, weight_t)

        cat_probs = F.softmax(out["category"], dim=-1)[0]
        cat_idx = int(cat_probs.argmax())
        category = EWASTE_CATEGORIES[cat_idx]

        # Restrict sub-category to those valid for the predicted category.
        sub_mask = torch.full((len(SUB_CATEGORIES),), float("-inf"), device=self.device)
        for name in CATEGORY_TO_SUB_CATEGORIES[category]:
            sub_mask[_SUB_TO_IDX[name]] = 0.0
        sub_probs = F.softmax(out["sub_category"][0] + sub_mask, dim=-1)
        sub_idx = int(sub_probs.argmax())
        sub_label: Optional[str] = SUB_CATEGORIES[sub_idx]
        sub_conf = float(sub_probs[sub_idx])
        if sub_conf < cfg.sub_category_min_confidence or (
            cfg.null_generic_sub_category and sub_label == GENERIC_SUB_CATEGORY
        ):
            sub_label = None

        cond_probs = F.softmax(out["condition"], dim=-1)[0]
        cond_idx = int(cond_probs.argmax())

        mat_probs = torch.sigmoid(out["material"])[0]
        detected = [m for m, p in zip(MATERIALS, mat_probs.tolist()) if p >= cfg.material_threshold]

        estimated_value = float(log_to_inr(out["estimated_value_log"])[0])

        if user_weight is not None:
            weight, weight_source = user_weight, "user_input"
        else:
            weight, weight_source = cfg.default_weight_prior_kg[category], "category_prior"

        core: Dict[str, Any] = {
            "pipeline_route": "E_WASTE",
            "material_category": category,
            "sub_category": sub_label,
            "approx_weight_kg": round(weight, 3),
            "weight_source": weight_source,
            "estimated_value": round(estimated_value, 2),
            "physical_condition": CONDITIONS[cond_idx],
            "detected_materials": detected,
            "material_probabilities": {m: round(p, 4) for m, p in zip(MATERIALS, mat_probs.tolist())},
            "confidence": {
                "segregation": round(p_ewaste, 4),
                "category": round(float(cat_probs[cat_idx]), 4),
                "sub_category": round(sub_conf, 4),
                "condition": round(float(cond_probs[cond_idx]), 4),
            },
        }
        return self._serialize(core)

    # -- response builders -------------------------------------------------------
    def _mixed_plastics_response(
        self, user_weight: Optional[float], conf_mixed: float, p_ewaste: float
    ) -> Dict[str, Any]:
        cfg = self.config
        if user_weight is not None:
            weight, weight_source = user_weight, "user_input"
        else:
            weight, weight_source = cfg.default_weight_prior_kg["MixedPlastics"], "category_prior"

        value: Optional[float] = None
        if cfg.mixed_plastics_rate_inr_per_kg is not None:
            value = round(cfg.mixed_plastics_rate_inr_per_kg * weight, 2)

        core: Dict[str, Any] = {
            "pipeline_route": "MIXED_PLASTICS",
            "material_category": "MixedPlastics",
            "sub_category": None,
            "approx_weight_kg": round(weight, 3),
            "weight_source": weight_source,
            "estimated_value": value,
            "physical_condition": None,
            "detected_materials": [],
            "material_probabilities": {},
            "confidence": {"segregation": round(conf_mixed, 4)},
        }
        return self._serialize(core)

    @staticmethod
    def _serialize(core: Dict[str, Any]) -> Dict[str, Any]:
        """Return the backend (snake_case) payload and the Dart UI (camelCase) state."""
        return {"rest_api": core, "dart_ui_state": _to_camel_keys(core)}


## 9. Execution verification

Smoke test with random weights. Predictions are only meaningful as a shape / schema check.

In [67]:
torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# pretrained=False keeps this smoke test offline; weights are random, so the
# predictions below are only meaningful as a shape / schema check.
seg_model = WasteSegregationCNN(backbone="mobilenet_v3_small", pretrained=False)
val_model = EwasteValuationMultiHeadCNN(pretrained=False)

dummy = torch.randn(1, 3, 224, 224)


Using device: cpu


In [68]:
# 1) Default routing
engine = KabadiwalaAIInferenceEngine(seg_model, val_model, device=device)
print("\n--- Default routing (weight = 1.25 kg) ---")
print(json.dumps(engine.predict(dummy, approx_weight_kg=1.25), indent=2))



--- Default routing (weight = 1.25 kg) ---
{
  "rest_api": {
    "pipeline_route": "E_WASTE",
    "material_category": "Cables",
    "sub_category": "Copper_Cable",
    "approx_weight_kg": 1.25,
    "weight_source": "user_input",
    "estimated_value": 0.0,
    "physical_condition": "Minor_Damage",
    "detected_materials": [
      "Copper",
      "Aluminum",
      "Lithium",
      "Precious_Metals_PCB"
    ],
    "material_probabilities": {
      "Copper": 0.5052,
      "Aluminum": 0.5016,
      "Gold_Plated_Connectors": 0.4917,
      "Lithium": 0.5057,
      "Steel_Iron": 0.4989,
      "Precious_Metals_PCB": 0.5077
    },
    "confidence": {
      "segregation": 0.5006,
      "category": 0.21,
      "sub_category": 0.5065,
      "condition": 0.3424
    }
  },
  "dart_ui_state": {
    "pipelineRoute": "E_WASTE",
    "category": "Cables",
    "subCategory": "Copper_Cable",
    "approxWeightKg": 1.25,
    "weightSource": "user_input",
    "estimatedValue": 0.0,
    "physicalCondition":

In [69]:
# 2) Force the E-waste route so Pipeline 2 is always exercised
force_cfg = EngineConfig(e_waste_threshold=0.0)
engine_forced = KabadiwalaAIInferenceEngine(seg_model, val_model, config=force_cfg, device=device)
print("\n--- Forced E_Waste route, no user weight ---")
print(json.dumps(engine_forced.predict(dummy), indent=2))



--- Forced E_Waste route, no user weight ---
{
  "rest_api": {
    "pipeline_route": "E_WASTE",
    "material_category": "Cables",
    "sub_category": "Copper_Cable",
    "approx_weight_kg": 1.0,
    "weight_source": "category_prior",
    "estimated_value": 0.0,
    "physical_condition": "Minor_Damage",
    "detected_materials": [
      "Copper",
      "Aluminum",
      "Lithium",
      "Precious_Metals_PCB"
    ],
    "material_probabilities": {
      "Copper": 0.5052,
      "Aluminum": 0.5016,
      "Gold_Plated_Connectors": 0.4917,
      "Lithium": 0.5057,
      "Steel_Iron": 0.4989,
      "Precious_Metals_PCB": 0.5077
    },
    "confidence": {
      "segregation": 0.5006,
      "category": 0.21,
      "sub_category": 0.5065,
      "condition": 0.3424
    }
  },
  "dart_ui_state": {
    "pipelineRoute": "E_WASTE",
    "category": "Cables",
    "subCategory": "Copper_Cable",
    "approxWeightKg": 1.0,
    "weightSource": "category_prior",
    "estimatedValue": 0.0,
    "physicalCon

In [70]:
# 3) Force the MixedPlastics route (with an example rule-based rate)
plastic_cfg = EngineConfig(e_waste_threshold=1.01, mixed_plastics_rate_inr_per_kg=10.0)
engine_plastic = KabadiwalaAIInferenceEngine(seg_model, val_model, config=plastic_cfg, device=device)
print("\n--- Forced MixedPlastics route (weight = 3 kg) ---")
print(json.dumps(engine_plastic.predict(dummy, approx_weight_kg=3.0), indent=2))



--- Forced MixedPlastics route (weight = 3 kg) ---
{
  "rest_api": {
    "pipeline_route": "MIXED_PLASTICS",
    "material_category": "MixedPlastics",
    "sub_category": null,
    "approx_weight_kg": 3.0,
    "weight_source": "user_input",
    "estimated_value": 30.0,
    "physical_condition": null,
    "detected_materials": [],
    "material_probabilities": {},
    "confidence": {
      "segregation": 0.4994
    }
  },
  "dart_ui_state": {
    "pipelineRoute": "MIXED_PLASTICS",
    "category": "MixedPlastics",
    "subCategory": null,
    "approxWeightKg": 3.0,
    "weightSource": "user_input",
    "estimatedValue": 30.0,
    "physicalCondition": null,
    "detectedMaterials": [],
    "materialProbabilities": {},
    "confidence": {
      "segregation": 0.4994
    }
  }
}


In [71]:
# 4) Loss / training-step sanity check on a fake batch
val_model.train()
batch = {
    "image": torch.randn(2, 3, 224, 224, device=device),
    "category": torch.tensor([0, 3], device=device),
    "sub_category": torch.tensor([0, IGNORE_INDEX], device=device),  # 2nd label missing
    "condition": torch.tensor([1, 2], device=device),
    "materials": torch.tensor([[1, 0, 1, 0, 0, 1], [0, 0, 0, 1, 0, 0]], dtype=torch.float32, device=device),
    "material_mask": torch.ones(2, device=device),
    "approx_weight_kg": torch.tensor([0.8, 0.0], device=device),
    "has_weight": torch.tensor([1.0, 0.0], device=device),
    "estimated_value": torch.tensor([240.0, 0.0], device=device),
    "value_mask": torch.tensor([1.0, 0.0], device=device),
}
criterion = EwasteMultiTaskLoss().to(device)
outputs = val_model(batch["image"], batch["approx_weight_kg"], batch["has_weight"])
total_loss, parts = criterion(outputs, batch)
total_loss.backward()
print("\n--- Loss sanity check ---")
print({k: round(float(v), 4) for k, v in parts.items()})



--- Loss sanity check ---
{'category': 1.6882, 'sub_category': 1.8086, 'condition': 1.0568, 'material': 0.6962, 'value': 5.0507, 'total': 6.9155}


# Part 2: Bootstrapping and training from zero

Sections 1-9 defined the architecture. Sections 10-16 get you a **first trained model without any private dataset**:

1. Pull public datasets (Roboflow Universe e-waste sets, TrashNet).
2. Train **Pipeline 1** (`MixedPlastics` vs `E_Waste`) on them.
3. Build **Pipeline 2** labels by weak supervision: category from the source dataset, `sub_category` / `condition` from zero-shot CLIP (low-confidence answers are ignored, which the masks in `EwasteDataset` already support), `materials` from category rules, and `estimatedValue` from a rate card x weight.
4. Train **Pipeline 2**, then plug both checkpoints into `KabadiwalaAIInferenceEngine`.

**Read this before trusting the output**

- **Pipeline 1 shortcut risk.** TrashNet photos are objects on a plain white board, while the e-waste sets are cluttered. The model can learn "white background = plastic". Validate on your own real photos, and add ~200 real `MixedPlastics` photos from actual scrap shops as soon as you can.
- **Category head** learns real visual signal, but only for categories the public sets cover. `CRT` and `Motor` may have little or no data. The notebook prints per-category counts and warns when they are low.
- **`sub_category` and `condition`** come from CLIP zero-shot and are noisy, especially `condition`. Treat them as a starting point to be corrected with human review.
- **`materials`** are rule-based from the category (a PCB "has" copper and gold-plated connectors). The head learns category-typical materials, not what is visibly present.
- **`estimatedValue`** is `rate x weight x condition multiplier`, using **placeholder** rates and **synthetic** weights (weight cannot be seen in a photo). The value head learns your rate card, not the real market. Replace the rates with your actual kabadiwala rate card, and later with real payouts.
- **Scores on public data overstate real-world accuracy.** Build a small test set from real photos taken in the field.

**Licences.** Several Roboflow sets are CC BY 4.0 (attribution required); check each project's page for its licence. TrashNet is MIT-licensed.

## 10. Configuration and data download

Run on a GPU runtime (Colab T4 is enough). You need a free [Roboflow](https://roboflow.com) API key in the `ROBOFLOW_API_KEY` environment variable. Open each project's Universe page and set `version` to the latest version number.

In [72]:
# Uncomment on the first run
#%pip install -q roboflow transformers pyyaml


In [73]:
import os
os.environ["ROBOFLOW_API_KEY"] = "xdT3uTf4M3SK8aIjGJnU"

In [74]:
import os
import re
import subprocess
import time
import zipfile
from collections import Counter, defaultdict

from torch.utils.data import DataLoader, WeightedRandomSampler

SEED = 42
WORK_DIR = Path("/content/drive/MyDrive/kabadiwala_work")
RAW_DIR = WORK_DIR / "raw"
CROP_DIR = WORK_DIR / "crops"
CKPT_DIR = WORK_DIR / "checkpoints"
for _d in (RAW_DIR, CROP_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

SEG_CKPT = CKPT_DIR / "segregation_best.pt"
VAL_CKPT = CKPT_DIR / "valuation_best.pt"
SEG_IMAGE_SIZE = 224
VAL_IMAGE_SIZE = 380  # lower to 320 or 288 if you run out of GPU memory (keep EngineConfig in sync)

ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "")

# Workspace / project ids come from the Universe URL: universe.roboflow.com/<workspace>/<project>.
# "version" is a guess: open the project page and use the latest version number.
ROBOFLOW_SOURCES: List[Dict[str, Any]] = [
    {"workspace": "student-esos5", "project": "e-waste-classifications", "version": 5},
    {"workspace": "student-esos5", "project": "e-waste-u7rro", "version": 3},
    {"workspace": "jensen", "project": "e-waste-detection-g7vf3", "version": 1},
    {"workspace": "work-9dvgk", "project": "ewaste-efxwy", "version": 1},
    {"workspace": "vision-experiments-fprmb", "project": "electronic-waste-object-detection-system", "version": 1},
]


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cpu


In [75]:
def download_roboflow(src: Dict[str, Any], dest: Path) -> Path:
    """Download one Roboflow project version in YOLOv8 format (images + labels + data.yaml)."""
    from roboflow import Roboflow

    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(src["workspace"]).project(src["project"])
    return Path(project.version(src["version"]).download("yolov8", location=str(dest)).location)


if ROBOFLOW_API_KEY:
    for src in ROBOFLOW_SOURCES:
        dest = RAW_DIR / f'{src["project"]}-v{src["version"]}'
        if dest.exists():
            continue
        try:
            download_roboflow(src, dest)
        except Exception as exc:  # wrong version / private project / network
            print(f'Could not download {src["project"]} v{src["version"]}: {exc}')
else:
    print("ROBOFLOW_API_KEY not set. Skipping downloads; YOLOv8 exports placed under", RAW_DIR, "are still used.")

RF_EXPORTS = sorted(p for p in RAW_DIR.glob("*") if (p / "data.yaml").exists())
print("Roboflow exports found:", [p.name for p in RF_EXPORTS])

# TrashNet (MIT): the "plastic" class gives the MixedPlastics negatives for Pipeline 1.
TRASHNET_DIR = RAW_DIR / "trashnet"
TRASHNET_IMAGES = TRASHNET_DIR / "data" / "dataset-resized"


def fetch_trashnet() -> None:
    if TRASHNET_IMAGES.exists():
        return
    if not TRASHNET_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/garythung/trashnet", str(TRASHNET_DIR)],
            check=True,
        )
    archive = TRASHNET_DIR / "data" / "dataset-resized.zip"
    if archive.exists():
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(TRASHNET_DIR / "data")
    else:
        print("dataset-resized.zip not found; download TrashNet manually into", TRASHNET_IMAGES)


try:
    fetch_trashnet()
except Exception as exc:
    print("TrashNet download failed:", exc)


Roboflow exports found: ['e-waste-classifications-v5', 'e-waste-detection-g7vf3-v1', 'e-waste-u7rro-v3', 'electronic-waste-object-detection-system-v1', 'ewaste-efxwy-v1']


## 11. Ingestion helpers

Roboflow detection exports contain boxes (or polygons). We use them twice: the **full images** become E_Waste positives for Pipeline 1, and the **cropped objects** become labelled examples for Pipeline 2. The source dataset's own train/valid/test split is preserved (Roboflow augmentations only touch `train`), which limits duplicate leakage between train and validation.

Roboflow class names vary per project. `map_class_to_category` maps by keyword and prints every class it could not map, so you can extend `EXTRA_CLASS_MAP`.

In [76]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_DIRS = {"train": "train", "valid": "val", "val": "val", "test": "val"}

_CLASS_KEYWORDS: Dict[str, set] = {
    "PCB": {"pcb", "pcba", "circuit", "motherboard", "mainboard", "gpu"},
    "Battery": {"battery", "batteries", "accumulator", "powerbank"},
    "Cables": {"cable", "cables", "wire", "wires", "cord", "cords"},
    "CRT": {"crt"},
    "Motor": {"motor", "motors"},
}
# Lower-case class name -> category, checked first. Example: {"circuit_board": "PCB"}
EXTRA_CLASS_MAP: Dict[str, str] = {}


def map_class_to_category(name: str) -> Optional[str]:
    """Map a dataset class name to a Data Dictionary category (or ``None`` if unrelated)."""
    key = name.strip().lower()
    if key in EXTRA_CLASS_MAP:
        return EXTRA_CLASS_MAP[key]
    tokens = {t for t in re.split(r"[^a-z0-9]+", key) if t}
    for category, words in _CLASS_KEYWORDS.items():
        if tokens & words:
            return category
    return None


def gather_yolo_images(export_dir: Path) -> List[Dict[str, Any]]:
    """All full images of a YOLO export as ``{"image_path", "split"}`` (Pipeline 1 positives)."""
    items = []
    for split_dir, split in SPLIT_DIRS.items():
        img_dir = Path(export_dir) / split_dir / "images"
        if img_dir.is_dir():
            items += [
                {"image_path": str(p), "split": split}
                for p in sorted(img_dir.iterdir())
                if p.suffix.lower() in IMG_EXTS
            ]
    return items


def crop_yolo_export(
    export_dir: Path,
    out_dir: Path,
    class_to_category: Callable[[str], Optional[str]] = map_class_to_category,
    min_side: int = 48,
    margin: float = 0.08,
) -> List[Dict[str, Any]]:
    """
    Crop every labelled object of a YOLOv8 export into ``out_dir/<split>/<category>/*.jpg``.

    Handles both box labels (``cls cx cy w h``) and polygon labels (``cls x1 y1 x2 y2 ...``,
    reduced to their bounding box). Returns ``{"image_path", "category", "split"}`` items.
    """
    import yaml

    export_dir, out_dir = Path(export_dir), Path(out_dir)
    with open(export_dir / "data.yaml", "r", encoding="utf-8") as fh:
        names = yaml.safe_load(fh)["names"]
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]

    items: List[Dict[str, Any]] = []
    unmapped: Counter = Counter()
    for split_dir, split in SPLIT_DIRS.items():
        img_dir, lbl_dir = export_dir / split_dir / "images", export_dir / split_dir / "labels"
        if not img_dir.is_dir():
            continue
        for img_path in sorted(img_dir.iterdir()):
            lbl_path = lbl_dir / f"{img_path.stem}.txt"
            if img_path.suffix.lower() not in IMG_EXTS or not lbl_path.exists():
                continue
            with Image.open(img_path) as im:
                image = im.convert("RGB")
            width, height = image.size
            for j, line in enumerate(lbl_path.read_text().splitlines()):
                parts = line.split()
                if len(parts) < 5:
                    continue
                cls_name = names[int(float(parts[0]))]
                category = class_to_category(cls_name)
                if category is None:
                    unmapped[cls_name] += 1
                    continue
                coords = [float(v) for v in parts[1:]]
                if len(coords) == 4:
                    cx, cy, w, h = coords
                    x0, y0, x1, y1 = cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2
                else:
                    xs, ys = coords[0::2], coords[1::2]
                    x0, y0, x1, y1 = min(xs), min(ys), max(xs), max(ys)
                mx, my = (x1 - x0) * margin, (y1 - y0) * margin
                box = (
                    max(0, int((x0 - mx) * width)), max(0, int((y0 - my) * height)),
                    min(width, int((x1 + mx) * width)), min(height, int((y1 + my) * height)),
                )
                if box[2] - box[0] < min_side or box[3] - box[1] < min_side:
                    continue
                dest = out_dir / split / category
                dest.mkdir(parents=True, exist_ok=True)
                out_path = dest / f"{export_dir.name}_{img_path.stem}_{j}.jpg"
                image.crop(box).save(out_path, quality=92)
                items.append({"image_path": str(out_path), "category": category, "split": split})

    if unmapped:
        print(f"[{export_dir.name}] unmapped classes (extend EXTRA_CLASS_MAP if any belong in a category):")
        print("   ", dict(unmapped.most_common(15)))
    return items


def index_class_folders(
    root: Path, class_to_category: Callable[[str], Optional[str]] = map_class_to_category
) -> List[Dict[str, Any]]:
    """
    Index ``root/[train|valid|test/]<class_name>/*.jpg`` trees (e.g. your own photos, TrashNet).
    ``split`` is ``None`` when the tree has no split folder (assigned later by ``finalize_splits``).
    """
    root = Path(root)
    items = []
    if not root.exists():
        return items
    for p in sorted(root.rglob("*")):
        if p.suffix.lower() not in IMG_EXTS:
            continue
        category = class_to_category(p.parent.name)
        if category is None:
            continue
        first = p.relative_to(root).parts[0]
        items.append({"image_path": str(p), "category": category, "split": SPLIT_DIRS.get(first)})
    return items


def finalize_splits(items: List[Dict[str, Any]], val_frac: float = 0.15, seed: int = SEED) -> None:
    """Randomly assign ``split`` where missing; if there is no val data at all, hold out ``val_frac``."""
    rng = random.Random(seed)
    for it in items:
        if it.get("split") is None:
            it["split"] = "val" if rng.random() < val_frac else "train"
    if items and not any(it["split"] == "val" for it in items):
        for it in items:
            it["split"] = "val" if rng.random() < val_frac else "train"


def cap_per_category(items: List[Dict[str, Any]], cap: int, seed: int = SEED) -> List[Dict[str, Any]]:
    """Limit each category to ``cap`` items (keeps CLIP labelling and training time bounded)."""
    rng, by_cat = random.Random(seed), defaultdict(list)
    for it in items:
        by_cat[it["category"]].append(it)
    out: List[Dict[str, Any]] = []
    for lst in by_cat.values():
        rng.shuffle(lst)
        out += lst[:cap]
    return out


In [77]:
# ---- Pipeline 1 items: {"image_path", "label", "split"}  (0 = MixedPlastics, 1 = E_Waste) ----
p1_items: List[Dict[str, Any]] = []
# ---- Pipeline 2 items: {"image_path", "category", "split"} ----
p2_items: List[Dict[str, Any]] = []

for export in RF_EXPORTS:
    p1_items += [{**it, "label": 1} for it in gather_yolo_images(export)]
    p2_items += crop_yolo_export(export, CROP_DIR)

plastics = index_class_folders(TRASHNET_IMAGES, lambda n: "MixedPlastics" if n == "plastic" else None)
p1_items += [{"image_path": it["image_path"], "split": it["split"], "label": 0} for it in plastics]

# Your own photos (recommended!): put them in my_photos/<class_name>/*.jpg, e.g. my_photos/pcb/1.jpg
# p2_items += index_class_folders("my_photos")

finalize_splits(p1_items)
finalize_splits(p2_items)

print("Pipeline 1:", Counter((it["label"], it["split"]) for it in p1_items))
p2_counts = Counter(it["category"] for it in p2_items if it["split"] == "train")
print("Pipeline 2 train crops per category:", dict(p2_counts))
for cat in EWASTE_CATEGORIES:
    if p2_counts.get(cat, 0) < 100:
        print(f"WARNING: only {p2_counts.get(cat, 0)} train images for '{cat}'. Add your own photos before relying on it.")


KeyboardInterrupt: 

## 12. Train Pipeline 1 (MixedPlastics vs E_Waste)

Class-weighted cross-entropy, AdamW + cosine schedule, mixed precision on GPU, backbone frozen for the first epochs, best checkpoint by validation loss with early stopping. We also report E_Waste precision and recall and pick a routing threshold that keeps recall high, since a missed e-waste item is the costlier error.

In [ ]:
class SegregationDataset(Dataset):
    """Pipeline 1 dataset over ``{"image_path", "label"}`` items."""

    def __init__(self, items: Sequence[Dict[str, Any]], image_size: int = 224, training: bool = True) -> None:
        self.items = list(items)
        self.transform = build_train_transforms(image_size) if training else build_eval_transforms(image_size)

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        item = self.items[idx]
        with Image.open(item["image_path"]) as im:
            arr = np.asarray(im.convert("RGB"))
        return self.transform(image=arr)["image"], torch.tensor(item["label"], dtype=torch.long)


def _to_device(batch: Dict[str, torch.Tensor], dev: torch.device) -> Dict[str, torch.Tensor]:
    return {k: v.to(dev, non_blocking=True) for k, v in batch.items()}


def _make_scaler(enabled: bool):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def set_trainable(module: nn.Module, flag: bool) -> None:
    for p in module.parameters():
        p.requires_grad = flag


@torch.inference_mode()
def evaluate_segregation(model: nn.Module, loader: DataLoader, criterion: nn.Module, dev: torch.device) -> Dict[str, float]:
    model.eval()
    use_amp = dev.type == "cuda"
    loss_sum, n, tp, fp, fn, correct = 0.0, 0, 0, 0, 0, 0
    for x, y in loader:
        x, y = x.to(dev), y.to(dev)
        with torch.autocast(device_type=dev.type, enabled=use_amp):
            logits = model(x)
        logits = logits.float()
        loss_sum += float(criterion(logits, y)) * x.size(0)
        pred = logits.argmax(1)
        n += x.size(0)
        correct += int((pred == y).sum())
        tp += int(((pred == 1) & (y == 1)).sum())
        fp += int(((pred == 1) & (y == 0)).sum())
        fn += int(((pred == 0) & (y == 1)).sum())
    return {
        "loss": loss_sum / max(n, 1),
        "acc": correct / max(n, 1),
        "ewaste_precision": tp / max(tp + fp, 1),
        "ewaste_recall": tp / max(tp + fn, 1),
    }


def train_segregation(
    model: WasteSegregationCNN,
    train_items: Sequence[Dict[str, Any]],
    val_items: Sequence[Dict[str, Any]],
    dev: torch.device,
    epochs: int = 8,
    batch_size: int = 64,
    lr: float = 3e-4,
    freeze_epochs: int = 2,
    patience: int = 4,
    num_workers: int = 2,
    image_size: int = 224,
    ckpt_path: Path = SEG_CKPT,
) -> List[Dict[str, float]]:
    """Train Pipeline 1; saves the best ``state_dict`` to ``ckpt_path`` and returns per-epoch metrics."""
    labels = np.array([it["label"] for it in train_items])
    if len(set(labels.tolist())) < 2 or len(val_items) == 0:
        raise ValueError("Need both classes in train and a non-empty validation set.")

    train_ds = SegregationDataset(train_items, image_size, training=True)
    val_ds = SegregationDataset(val_items, image_size, training=False)
    pin = dev.type == "cuda"
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                              pin_memory=pin, drop_last=len(train_ds) > batch_size)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin)

    counts = np.bincount(labels, minlength=2)
    class_w = torch.tensor(counts.sum() / (2.0 * np.maximum(counts, 1)), dtype=torch.float32, device=dev)
    criterion = nn.CrossEntropyLoss(weight=class_w, label_smoothing=0.05)
    val_criterion = nn.CrossEntropyLoss()

    model.to(dev)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    use_amp = dev.type == "cuda"
    scaler = _make_scaler(use_amp)

    best, bad_epochs, history = float("inf"), 0, []
    for epoch in range(epochs):
        backbone_on = epoch >= freeze_epochs
        set_trainable(model.net.features, backbone_on)
        model.train()
        if not backbone_on:
            model.net.features.eval()  # keep frozen BatchNorm statistics fixed
        start, run_loss, seen = time.time(), 0.0, 0
        for x, y in train_loader:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=dev.type, enabled=use_amp):
                logits = model(x)
            loss = criterion(logits.float(), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            run_loss += float(loss) * x.size(0)
            seen += x.size(0)
        scheduler.step()

        metrics = evaluate_segregation(model, val_loader, val_criterion, dev)
        metrics.update({"epoch": epoch + 1, "train_loss": run_loss / max(seen, 1)})
        history.append(metrics)
        print(f"[P1 {epoch + 1:02d}/{epochs}] train {metrics['train_loss']:.4f} | val {metrics['loss']:.4f} "
              f"acc {metrics['acc']:.3f} E-waste P {metrics['ewaste_precision']:.3f} R {metrics['ewaste_recall']:.3f} "
              f"({time.time() - start:.0f}s)")
        if metrics["loss"] < best:
            best, bad_epochs = metrics["loss"], 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print("Early stopping.")
                break
    model.load_state_dict(torch.load(ckpt_path, map_location=dev))
    return history


@torch.inference_mode()
def pick_ewaste_threshold(model: WasteSegregationCNN, items: Sequence[Dict[str, Any]], dev: torch.device,
                          target_recall: float = 0.95, image_size: int = 224) -> float:
    """Highest P(E_Waste) threshold that still reaches ``target_recall`` on the validation items."""
    loader = DataLoader(SegregationDataset(items, image_size, training=False), batch_size=64, num_workers=2)
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        probs.append(F.softmax(model(x.to(dev)).float(), dim=-1)[:, 1].cpu())
        labels.append(y)
    probs, labels = torch.cat(probs), torch.cat(labels)
    positives = max(int((labels == 1).sum()), 1)
    for thr in np.arange(0.95, 0.04, -0.05):
        recall = int(((probs >= thr) & (labels == 1)).sum()) / positives
        if recall >= target_recall:
            return float(round(thr, 2))
    return 0.05


In [ ]:
p1_train = [it for it in p1_items if it["split"] == "train"]
p1_val = [it for it in p1_items if it["split"] == "val"]

seg_model = WasteSegregationCNN(backbone="mobilenet_v3_small", pretrained=True)
seg_history = train_segregation(seg_model, p1_train, p1_val, device)

E_WASTE_THRESHOLD = pick_ewaste_threshold(seg_model, p1_val, device)
print("Suggested e_waste_threshold:", E_WASTE_THRESHOLD)


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 106MB/s]
Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)


[P1 01/8] train 0.6167 | val 0.3766 acc 0.905 E-waste P 1.000 R 0.903 (84s)
[P1 02/8] train 0.5425 | val 0.4045 acc 0.910 E-waste P 1.000 R 0.907 (79s)
[P1 03/8] train 0.5338 | val 0.3384 acc 0.987 E-waste P 1.000 R 0.987 (87s)
[P1 04/8] train 0.5078 | val 0.4206 acc 0.987 E-waste P 1.000 R 0.986 (85s)
[P1 05/8] train 0.5069 | val 0.3773 acc 0.995 E-waste P 1.000 R 0.995 (84s)
[P1 06/8] train 0.4890 | val 0.3947 acc 0.993 E-waste P 1.000 R 0.992 (84s)
[P1 07/8] train 0.4929 | val 0.3659 acc 0.997 E-waste P 1.000 R 0.997 (84s)
Early stopping.
Suggested e_waste_threshold: 0.55


## 13. Weak labels for Pipeline 2

- **Category**: from the source dataset (trusted). Crops where CLIP gives the stated category almost no probability are dropped as likely label noise.
- **`sub_category` / `condition`**: zero-shot CLIP. Only answers above a confidence threshold are kept; the rest become "unknown" and are skipped by the loss.
- **`materials`**: category rules (crude, see the caveats above).
- **`estimated_value`**: `RuleBasedValuator` = rate per kg x condition multiplier x weight. Weights are **synthetic** (lognormal around the category prior), because a photo does not show weight.

**Replace `PLACEHOLDER_RATE_INR_PER_KG` with your real rate card before using any value output.**

In [ ]:
def _as_tensor(x: Any) -> torch.Tensor:
    """Different `transformers` versions return a tensor or an output object from get_*_features."""
    if isinstance(x, torch.Tensor):
        return x
    for attr in ("pooler_output", "image_embeds", "text_embeds"):
        if getattr(x, attr, None) is not None:
            return getattr(x, attr)
    raise TypeError(f"Unexpected CLIP output type: {type(x)}")


CLIP_CATEGORY_PROMPTS = {
    "PCB": "a photo of a printed circuit board",
    "CRT": "a photo of an old CRT monitor or television",
    "Cables": "a photo of a bundle of electrical cables and wires",
    "Battery": "a photo of a battery",
    "Motor": "a photo of an electric motor",
}
CLIP_SUB_PROMPTS = {
    "Motherboard": "a photo of a computer motherboard",
    "GPU_Board": "a photo of a graphics card",
    "Power_Supply": "a photo of a computer power supply unit",
    "Li_Ion_Cell": "a photo of a lithium-ion battery cell",
    "Copper_Cable": "a photo of copper wire cables",
    "General_Ewaste": "a photo of a piece of electronic waste",
}
CLIP_CONDITION_PROMPTS = {
    "Intact": "a photo of an intact electronic device in good condition",
    "Minor_Damage": "a photo of a slightly damaged, dusty or scratched electronic device",
    "Scrap_Only": "a photo of broken, burnt, corroded or dismantled electronic scrap",
}


class ClipPseudoLabeler:
    """Zero-shot image labeller built on a Hugging Face CLIP model."""

    def __init__(self, dev: torch.device, model_name: str = "openai/clip-vit-base-patch32") -> None:
        from transformers import CLIPModel, CLIPProcessor

        self.device = dev
        self.model = CLIPModel.from_pretrained(model_name).to(dev).eval()
        self.processor = CLIPProcessor.from_pretrained(model_name)

    @torch.inference_mode()
    def _encode_text(self, prompts: Sequence[str]) -> torch.Tensor:
        tokens = self.processor(text=list(prompts), return_tensors="pt", padding=True).to(self.device)
        feats = _as_tensor(self.model.get_text_features(**tokens))
        return feats / feats.norm(dim=-1, keepdim=True)

    @torch.inference_mode()
    def score(self, image_paths: Sequence[str], prompt_sets: Dict[str, Sequence[str]], batch_size: int = 32) -> Dict[str, np.ndarray]:
        """Return ``{name: (N, K) softmax probabilities}`` for every prompt set."""
        text = {name: self._encode_text(prompts) for name, prompts in prompt_sets.items()}
        out: Dict[str, List[np.ndarray]] = {name: [] for name in prompt_sets}
        for i in range(0, len(image_paths), batch_size):
            images = []
            for path in image_paths[i:i + batch_size]:
                with Image.open(path) as im:
                    images.append(im.convert("RGB"))
            pixels = self.processor(images=images, return_tensors="pt")["pixel_values"].to(self.device)
            feats = _as_tensor(self.model.get_image_features(pixel_values=pixels))
            feats = feats / feats.norm(dim=-1, keepdim=True)
            for name, txt in text.items():
                out[name].append((100.0 * feats @ txt.T).softmax(dim=-1).cpu().numpy())
        return {
            name: (np.concatenate(chunks, axis=0) if chunks else np.zeros((0, len(prompt_sets[name]))))
            for name, chunks in out.items()
        }


# PLACEHOLDER rate card in INR per kg. Replace with your real kabadiwala rates.
PLACEHOLDER_RATE_INR_PER_KG: Dict[str, float] = {
    "PCB": 250.0, "CRT": 15.0, "Cables": 150.0, "Battery": 60.0, "Motor": 150.0, "MixedPlastics": 12.0,
}
CONDITION_MULTIPLIER: Dict[str, float] = {"Intact": 1.0, "Minor_Damage": 0.75, "Scrap_Only": 0.5}


class RuleBasedValuator:
    """Rule-based INR estimate: ``rate_per_kg[category] x condition_multiplier x weight_kg``."""

    def __init__(self, rate_inr_per_kg: Optional[Dict[str, float]] = None,
                 condition_multiplier: Optional[Dict[str, float]] = None) -> None:
        self.rate = dict(rate_inr_per_kg or PLACEHOLDER_RATE_INR_PER_KG)
        self.cond_mult = dict(condition_multiplier or CONDITION_MULTIPLIER)

    def estimate(self, category: str, condition: Optional[str], weight_kg: float) -> float:
        mult = self.cond_mult.get(condition or "Minor_Damage", self.cond_mult["Minor_Damage"])
        return round(self.rate[category] * mult * weight_kg, 2)


_BASE_MATERIALS: Dict[str, List[str]] = {
    "PCB": ["Copper", "Gold_Plated_Connectors", "Precious_Metals_PCB"],
    "CRT": ["Copper", "Steel_Iron"],
    "Cables": ["Copper"],
    "Battery": ["Steel_Iron"],
    "Motor": ["Copper", "Steel_Iron", "Aluminum"],
}


def material_rules(category: str, sub_category: Optional[str]) -> List[str]:
    """Category-typical materials (weak labels, not visual ground truth)."""
    mats = list(_BASE_MATERIALS[category])
    if sub_category == "Li_Ion_Cell":
        mats += ["Lithium", "Aluminum"]
    return sorted(set(mats))


def build_bootstrap_records(
    items: Sequence[Dict[str, Any]],
    labeler: ClipPseudoLabeler,
    valuator: RuleBasedValuator,
    sub_min_conf: float = 0.5,
    cond_min_conf: float = 0.6,
    verify_category: bool = True,
    min_category_prob: float = 0.05,
    seed: int = SEED,
) -> List[Dict[str, Any]]:
    """Turn ``{"image_path", "category", "split"}`` items into EwasteDataset records (snake_case)."""
    rng = np.random.default_rng(seed)
    scores = labeler.score(
        [it["image_path"] for it in items],
        {
            "category": [CLIP_CATEGORY_PROMPTS[c] for c in EWASTE_CATEGORIES],
            "sub": [CLIP_SUB_PROMPTS[s] for s in SUB_CATEGORIES],
            "cond": [CLIP_CONDITION_PROMPTS[c] for c in CONDITIONS],
        },
    )
    records: List[Dict[str, Any]] = []
    dropped = 0
    for i, it in enumerate(items):
        cat = it["category"]
        if verify_category and scores["category"][i][_CATEGORY_TO_IDX[cat]] < min_category_prob:
            dropped += 1
            continue

        allowed = CATEGORY_TO_SUB_CATEGORIES[cat]
        sub: Optional[str] = None
        if len(allowed) == 1:
            sub = allowed[0]
        else:
            idxs = [_SUB_TO_IDX[s] for s in allowed]
            probs = scores["sub"][i][idxs]
            probs = probs / probs.sum()
            best = int(probs.argmax())
            if probs[best] >= sub_min_conf:
                sub = allowed[best]

        cond_probs = scores["cond"][i]
        cond_idx = int(cond_probs.argmax())
        cond: Optional[str] = CONDITIONS[cond_idx] if cond_probs[cond_idx] >= cond_min_conf else None

        weight = round(DEFAULT_WEIGHT_PRIOR_KG[cat] * float(np.exp(rng.normal(0.0, 0.5))), 3)
        records.append({
            "image_path": it["image_path"],
            "split": it["split"],
            "material_category": cat,
            "sub_category": sub,
            "condition": cond,
            "materials": material_rules(cat, sub),
            "approx_weight_kg": weight,          # synthetic
            "estimated_value": valuator.estimate(cat, cond, weight),  # rate-card based
        })
    print(f"Built {len(records)} records ({dropped} dropped by CLIP category check).")
    return records


In [ ]:
MAX_TRAIN_PER_CATEGORY = 1500  # bounds CLIP labelling and training time
MAX_VAL_PER_CATEGORY = 300

p2_train_items = cap_per_category([it for it in p2_items if it["split"] == "train"], MAX_TRAIN_PER_CATEGORY)
p2_val_items = cap_per_category([it for it in p2_items if it["split"] == "val"], MAX_VAL_PER_CATEGORY)

labeler = ClipPseudoLabeler(device)
valuator = RuleBasedValuator()
train_records = build_bootstrap_records(p2_train_items, labeler, valuator)
val_records = build_bootstrap_records(p2_val_items, labeler, valuator)

with open(WORK_DIR / "records_train.json", "w", encoding="utf-8") as fh:
    json.dump(train_records, fh, indent=1)
with open(WORK_DIR / "records_val.json", "w", encoding="utf-8") as fh:
    json.dump(val_records, fh, indent=1)

print("label coverage (train):",
      {"sub_category": sum(r["sub_category"] is not None for r in train_records),
       "condition": sum(r["condition"] is not None for r in train_records),
       "total": len(train_records)})

del labeler  # free GPU memory before training Pipeline 2
if device.type == "cuda":
    torch.cuda.empty_cache()


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Built 2808 records (38 dropped by CLIP category check).
Built 677 records (5 dropped by CLIP category check).
label coverage (train): {'sub_category': 2494, 'condition': 1764, 'total': 2808}


## 14. Train Pipeline 2 (multi-head)

- Sampler weights by `1/sqrt(class count)` to soften category imbalance.
- The backbone gets a lower learning rate (0.3x) than the heads and is frozen for the first epochs.
- Mixed precision on GPU, gradient clipping, best checkpoint by validation loss, early stopping.
- Reports accuracy per head, material micro-F1 and value error in INR (MAE and median absolute % error).

Lower `VAL_IMAGE_SIZE` or the batch size if you hit GPU out-of-memory errors.

In [ ]:
@torch.inference_mode()
def evaluate_valuation(model: EwasteValuationMultiHeadCNN, loader: DataLoader, criterion: EwasteMultiTaskLoss,
                       dev: torch.device) -> Dict[str, float]:
    """Validation loss terms plus per-head metrics (unlabelled targets are skipped)."""
    model.eval()
    use_amp = dev.type == "cuda"
    sums: Dict[str, float] = defaultdict(float)
    n = cat_ok = sub_ok = sub_n = cond_ok = cond_n = tp = fp = fn = 0
    abs_err: List[torch.Tensor] = []
    pct_err: List[torch.Tensor] = []
    for batch in loader:
        batch = _to_device(batch, dev)
        with torch.autocast(device_type=dev.type, enabled=use_amp):
            out = model(batch["image"], batch["approx_weight_kg"], batch["has_weight"])
        out = {k: v.float() for k, v in out.items()}
        _, parts = criterion(out, batch)
        bs = batch["image"].size(0)
        for k, v in parts.items():
            sums[k] += float(v) * bs
        n += bs

        cat_ok += int((out["category"].argmax(1) == batch["category"]).sum())
        m = batch["sub_category"] != IGNORE_INDEX
        sub_ok += int((out["sub_category"].argmax(1)[m] == batch["sub_category"][m]).sum())
        sub_n += int(m.sum())
        m = batch["condition"] != IGNORE_INDEX
        cond_ok += int((out["condition"].argmax(1)[m] == batch["condition"][m]).sum())
        cond_n += int(m.sum())

        mm = batch["material_mask"].bool()
        pred, tgt = (torch.sigmoid(out["material"]) >= 0.5)[mm], batch["materials"][mm] > 0.5
        tp += int((pred & tgt).sum())
        fp += int((pred & ~tgt).sum())
        fn += int((~pred & tgt).sum())

        vm = batch["value_mask"].bool()
        if vm.any():
            pred_inr, tgt_inr = log_to_inr(out["estimated_value_log"][vm]), batch["estimated_value"][vm]
            abs_err.append((pred_inr - tgt_inr).abs().cpu())
            pct_err.append(((pred_inr - tgt_inr).abs() / tgt_inr.clamp(min=1.0)).cpu())

    metrics = {k: v / max(n, 1) for k, v in sums.items()}
    metrics.update({
        "category_acc": cat_ok / max(n, 1),
        "sub_category_acc": sub_ok / max(sub_n, 1),
        "condition_acc": cond_ok / max(cond_n, 1),
        "material_f1": 2 * tp / max(2 * tp + fp + fn, 1),
        "value_mae_inr": float(torch.cat(abs_err).mean()) if abs_err else float("nan"),
        "value_median_pct_err": float(torch.cat(pct_err).median()) if pct_err else float("nan"),
    })
    return metrics


def train_valuation(
    model: EwasteValuationMultiHeadCNN,
    train_ds: EwasteDataset,
    val_ds: EwasteDataset,
    dev: torch.device,
    epochs: int = 15,
    batch_size: int = 16,
    lr: float = 3e-4,
    freeze_epochs: int = 3,
    patience: int = 5,
    num_workers: int = 2,
    grad_clip: float = 1.0,
    criterion: Optional[EwasteMultiTaskLoss] = None,
    ckpt_path: Path = VAL_CKPT,
) -> List[Dict[str, float]]:
    """Train Pipeline 2; saves the best ``state_dict`` to ``ckpt_path`` and returns per-epoch metrics."""
    criterion = (criterion or EwasteMultiTaskLoss()).to(dev)
    model.to(dev)

    cat_of = [(r.get("material_category") or r.get("category")) for r in train_ds.records]
    freq = Counter(cat_of)
    weights = [1.0 / np.sqrt(freq[c]) for c in cat_of]
    sampler = WeightedRandomSampler(weights, num_samples=len(train_ds), replacement=True)
    pin = dev.type == "cuda"
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=num_workers,
                              pin_memory=pin, drop_last=len(train_ds) > batch_size)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin)

    head_params = [p for n, p in model.named_parameters() if not n.startswith("features.")]
    optimizer = torch.optim.AdamW(
        [{"params": model.features.parameters(), "lr": lr * 0.3}, {"params": head_params, "lr": lr}],
        weight_decay=1e-4,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    use_amp = dev.type == "cuda"
    scaler = _make_scaler(use_amp)

    best, bad_epochs, history = float("inf"), 0, []
    for epoch in range(epochs):
        backbone_on = epoch >= freeze_epochs
        set_trainable(model.features, backbone_on)
        model.train()
        if not backbone_on:
            model.features.eval()
        start, run_loss, seen = time.time(), 0.0, 0
        for batch in train_loader:
            batch = _to_device(batch, dev)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=dev.type, enabled=use_amp):
                out = model(batch["image"], batch["approx_weight_kg"], batch["has_weight"])
            out = {k: v.float() for k, v in out.items()}
            loss, _ = criterion(out, batch)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
            run_loss += float(loss) * batch["image"].size(0)
            seen += batch["image"].size(0)
        scheduler.step()

        metrics = evaluate_valuation(model, val_loader, criterion, dev)
        metrics.update({"epoch": epoch + 1, "train_loss": run_loss / max(seen, 1)})
        history.append(metrics)
        print(f"[P2 {epoch + 1:02d}/{epochs}] train {metrics['train_loss']:.4f} | val {metrics['total']:.4f} | "
              f"cat {metrics['category_acc']:.3f} sub {metrics['sub_category_acc']:.3f} cond {metrics['condition_acc']:.3f} "
              f"mat-F1 {metrics['material_f1']:.3f} | value MAE Rs{metrics['value_mae_inr']:.1f} "
              f"(median {100 * metrics['value_median_pct_err']:.0f}%) ({time.time() - start:.0f}s)")
        if metrics["total"] < best:
            best, bad_epochs = metrics["total"], 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print("Early stopping.")
                break
    model.load_state_dict(torch.load(ckpt_path, map_location=dev))
    return history


In [ ]:
train_ds = EwasteDataset(train_records, image_size=VAL_IMAGE_SIZE, training=True)
val_ds = EwasteDataset(val_records, image_size=VAL_IMAGE_SIZE, training=False)

val_model = EwasteValuationMultiHeadCNN(pretrained=True)
val_history = train_valuation(val_model, train_ds, val_ds, device)


Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 141MB/s]


[P2 01/15] train 2.2671 | val 1.3871 | cat 0.994 sub 0.805 cond 0.770 mat-F1 0.977 | value MAE Rs34.0 (median 46%) (60s)
[P2 02/15] train 1.3579 | val 1.2583 | cat 0.997 sub 0.836 cond 0.819 mat-F1 0.977 | value MAE Rs27.2 (median 37%) (48s)
[P2 03/15] train 1.2866 | val 1.1661 | cat 0.999 sub 0.858 cond 0.824 mat-F1 0.982 | value MAE Rs24.6 (median 34%) (49s)
[P2 04/15] train 1.7888 | val 1.2563 | cat 0.991 sub 0.841 cond 0.815 mat-F1 0.971 | value MAE Rs24.7 (median 35%) (81s)
[P2 05/15] train 1.3845 | val 1.2144 | cat 0.993 sub 0.838 cond 0.822 mat-F1 0.974 | value MAE Rs24.3 (median 34%) (73s)
[P2 06/15] train 1.3042 | val 1.2223 | cat 0.991 sub 0.849 cond 0.812 mat-F1 0.973 | value MAE Rs21.7 (median 32%) (73s)
[P2 07/15] train 1.1908 | val 1.1757 | cat 0.993 sub 0.854 cond 0.824 mat-F1 0.975 | value MAE Rs20.0 (median 31%) (71s)
[P2 08/15] train 1.1613 | val 1.1661 | cat 0.993 sub 0.846 cond 0.817 mat-F1 0.974 | value MAE Rs19.4 (median 29%) (72s)
Early stopping.


## 15. Plug the trained models into the inference engine

The engine loads both checkpoints and uses the recall-oriented routing threshold from section 12. Try it on real photos (not the training crops) to see how it behaves.

In [ ]:
engine_cfg = EngineConfig(
    e_waste_threshold=E_WASTE_THRESHOLD,
    valuation_image_size=VAL_IMAGE_SIZE,
    segregation_image_size=SEG_IMAGE_SIZE,
    # mixed_plastics_rate_inr_per_kg=PLACEHOLDER_RATE_INR_PER_KG["MixedPlastics"],  # optional
)
trained_engine = KabadiwalaAIInferenceEngine.from_checkpoints(
    segregation_ckpt=SEG_CKPT,
    valuation_ckpt=VAL_CKPT,
    config=engine_cfg,
    device=device,
)

sample_path = next(it["image_path"] for it in p1_val if it["label"] == 1)
print(json.dumps(trained_engine.predict(sample_path, approx_weight_kg=1.0), indent=2))


{
  "rest_api": {
    "pipeline_route": "E_WASTE",
    "material_category": "Battery",
    "sub_category": "Li_Ion_Cell",
    "approx_weight_kg": 1.0,
    "weight_source": "user_input",
    "estimated_value": 5.38,
    "physical_condition": "Intact",
    "detected_materials": [
      "Aluminum",
      "Lithium",
      "Steel_Iron"
    ],
    "material_probabilities": {
      "Copper": 0.0059,
      "Aluminum": 0.9427,
      "Gold_Plated_Connectors": 0.0009,
      "Lithium": 0.9388,
      "Steel_Iron": 0.9945,
      "Precious_Metals_PCB": 0.0009
    },
    "confidence": {
      "segregation": 0.7304,
      "category": 0.9138,
      "sub_category": 0.9661,
      "condition": 0.849
    }
  },
  "dart_ui_state": {
    "pipelineRoute": "E_WASTE",
    "category": "Battery",
    "subCategory": "Li_Ion_Cell",
    "approxWeightKg": 1.0,
    "weightSource": "user_input",
    "estimatedValue": 5.38,
    "physicalCondition": "Intact",
    "detectedMaterials": [
      "Aluminum",
      "Lithium",
 

## 16. What to do next (in this order)

1. **Collect real photos.** Have partner kabadiwalas photograph items through the app, with the category, the real weight and the payout they actually gave. Even 100-300 per category makes a big difference, and this becomes your honest test set.
2. **Swap the placeholder rate card** for your real one immediately, and retrain the value head with real payouts as soon as you have them (replace `estimated_value` in the records; keep everything else).
3. **Review the weak labels.** Sample about 100 records per category from `records_train.json`, fix wrong `sub_category` / `condition`, and add human-verified `materials`. That is the cheapest large quality gain.
4. **Re-check Pipeline 1 on real scrap-shop photos** for the white-background shortcut, and add real `MixedPlastics` images if needed.
5. **Fill category gaps** (`CRT`, `Motor`) with your own images or by adding more Roboflow projects via `ROBOFLOW_SOURCES` / `EXTRA_CLASS_MAP`.